In [1]:
import warnings
warnings.filterwarnings("ignore")
from langchain.llms import OpenAI
from langchain.chat_models import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.environ["OPENAI_API_KEY"]

In [2]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.chat import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import ChatOpenAI

template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("你是{product}的运维人员，你对{product}的上的开源项目都非常清楚。"),
        HumanMessagePromptTemplate.from_template("{resource}的开源项目是做什么？他的应用场景是？"),
    ]
)
llm =ChatOpenAI()
llm(template.format_messages(
    product="github",
    resource="langchain"
),
temperature=0.1
)

AIMessage(content='Langchain是一个开源的区块链项目，旨在为多语言编程提供一种新的解决方案。它的目标是解决不同编程语言之间的互操作性问题，使得不同语言编写的智能合约能够在区块链上无缝运行。\n\nLangchain的应用场景主要包括以下几个方面：\n1. 多语言智能合约开发：Langchain提供了一种统一的编程模型，使得开发人员可以使用不同的编程语言编写智能合约，并在区块链上运行。\n2. 跨链交互：Langchain可以实现不同区块链之间的互操作性，使得不同链上的智能合约能够相互调用和交互。\n3. 跨平台应用开发：Langchain可以支持在不同平台上运行的应用程序，使得开发人员可以使用不同的编程语言和框架来构建跨平台的应用。\n\n总之，Langchain的目标是提供一种简单、灵活和可扩展的解决方案，以促进多语言编程在区块链领域的应用和发展。', additional_kwargs={}, example=False)

In [3]:
# OutputParser
# Pydantic (JSON) Parser
# 自动根据Pydantic类的定义，生成输出的格式说明

from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel,Field, validator
from typing import List, Dict
import json

#避免中文变成unicode码
def chinese_fridendly(string):
    lines = string.split("\n")
    for i ,line in enumerate(lines):
        if line.startswith("{") and line.endswith("}"):
            try:
                lines[i] = json.dumps(json.loads(line), ensure_ascii=False, indent=2)
            except:
                pass
    return "\n".join(lines)

model_name="gpt-3.5-turbo-0613"
temperature=0.01
model =OpenAI(model_name=model_name,temperature=temperature)


#定义指定的输出格式类
class Command(BaseModel):
    command: str=Field(description="linux shell的命令名称")
    arguements: Dict[str,str]=Field(description="命令的参数 (name: value),name为执行命令，value为命令对应的参数值")

    #添加自定义的校验规则
    @validator("command")
    def no_space(cls, v):
        if " " in v or "\t" in v or "\n" in v:
            raise ValueError("命令名称不能包含空格或回车~")
        return v
    
#根据pydantic构建一个OutputParser
output_parser = PydanticOutputParser(pydantic_object=Command)

prompt = PromptTemplate(
    template="""将用户的指令转换成linux shell的命令.
    {format_instructions}
    {query}""",
    input_variables=["query"],
    #预先对模板的变量format_instructions做替换赋值
    partial_variables={"format_instructions": output_parser.get_format_instructions()}
)

# print(f"""------------------format_instructions------------------
# {chinese_fridendly(output_parser.get_format_instructions())}""")

query = "根据'ERROR'关键字过滤日志文件gateway.log"
input_model=prompt.format_prompt(query=query)

print(f"""------------------prompt------------------
{chinese_fridendly(input_model.to_string())}""")

output = model(input_model.to_string())
print(f"""------------------output------------------
{chinese_fridendly(output)}""")

cmd=output_parser.parse(output)
print(f"""------------------parse------------------
{cmd}""")




------------------prompt------------------
将用户的指令转换成linux shell的命令.
    The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{
  "properties": {
    "command": {
      "title": "Command",
      "description": "linux shell的命令名称",
      "type": "string"
    },
    "arguements": {
      "title": "Arguements",
      "description": "命令的参数 (name: value),name为执行命令，value为命令对应的参数值",
      "type": "object",
      "additionalProperties": {
        "type": "string"
      }
    }
  },
  "required": [
    "command",
    "arguements"
  ]
}
```
    根据'ERROR'关键字过滤日志文件gateway.log
------------------output---------

In [27]:
# Auto-Fixing Parser
# 利用LLM自动根据解析异常修复并重新解析,可以用来规避GPT吐回的内容格式不对的情况
from langchain.output_parsers import OutputFixingParser
new_parse=OutputFixingParser.from_llm(parser=output_parser, llm=ChatOpenAI())

#将之前的输出内容格式修改下
output = output.replace("}", " ")
print(f"""-----------worng-format--------
      {output}
""")

try:
    cmd=output_parser.parse(output)
except Exception as e:
    print(f"""-----------parse-error--------
          {e}
    """)
    cmd=new_parse.parse(output)
print(f"""----------cmd--------
        {cmd}
""")

-----------worng-format--------
      {
  "command": "grep",
  "arguements": {
    "ERROR": "gateway.log"
   
 

-----------parse-error--------
          Failed to parse Command from completion {
  "command": "grep",
  "arguements": {
    "ERROR": "gateway.log"
   
 . Got: Expecting value: line 1 column 1 (char 0)
    
----------cmd--------
        command='grep' arguements={'name': 'ERROR', 'value': 'gateway.log'}

